In [1]:
import os
from pathlib import Path  ## used to read path of file 
from dotenv import load_dotenv  ## load environment variable from .env file
from rich import print   ## display beautiful and formate output 
from langchain_core.tools import tool ## Convert a normal Python function into a LangChain tool. 
from langchain_core.messages import HumanMessage,ToolMessage ##Represent different types of messages in a LangChain conversation.
from langchain.chat_models import init_chat_model  ## Initialize a chat model using LangChain.


In [2]:
load_dotenv()
MODEL=os.getenv("gemini model","gemini-3.5-flash")


In [3]:
RESUME_DIR = (Path().cwd().parent / "resume")  ## used to get resume folder in ep 4 calling  
RESUME_DIR

WindowsPath('c:/Users/karti/agentic_ai_llm/ep_4_tool_callling/resume')

In [4]:
@tool
def add(a:float,b:float) -> float:
    """add two number together and return the addition result"""
    return a + b

In [5]:
@tool
def get_weather(city :str)-> str:
    """ get the current whether of city name """
    weather={"mumbai" : "32c, humid", "delhi" : "34c, cloudy", "nagpur" :"23c, rainy"}
    return f"{city} - {weather.get(city.lower(),f"no weather data for '{city}")}"

In [7]:
from langchain_core.tools import tool

@tool
def get_resume(name: str) -> str:
    """Read and return stored resume for candidate name (e.g., 'arav')."""
    first_name = name.lower().strip().split()[0]
    file_path = RESUME_DIR / f"{first_name}.txt"
    if file_path.exists():
        return file_path.read_text(encoding="utf-8")
    return f"no resume found for {name}"

In [8]:
print(get_resume.invoke("arav"))  

Aarav Sharma is a Software Developer based in Nagpur, Maharashtra. 
He can be contacted at aarav.sharma@example.com or 9000000001.
He completed his Bachelor of Technology in Computer Science 
from ABC Institute of Technology in 2024 with 82% marks. 
He has two years of experience working at Tech Solutions Pvt Ltd 
as a Software Developer. His responsibilities include developing 
applications, debugging software, writing database queries, and 
maintaining existing systems. He has technical skills in
 Python, Java, SQL, HTML, CSS, and Git. He developed a Student 
 Management System using Python and MySQL. He is interested in 
 software development and backend programming opportunities.

In [9]:
TOOLS=[add ,get_weather,lookup_user,get_resume]

In [ ]:
TOOLS_DATA={"add":add ,
             "get_weather":get_weather,
             "lookup_user": lookup_user,
             "get_resume": get_resume
            }

In [15]:
llm = init_chat_model(model=MODEL, model_provider="google_genai", temperature=0)
llm_with_tools=llm.bind_tools(TOOLS)

In [26]:
question="can we select arav for python developer in interview"
message=[HumanMessage(content=question)]
print (f"[bold cyan] USER :[/bold cyan] {question}\n")

 USER : can we select arav for python developer in interview

In [27]:
message

[HumanMessage(content='can we select arav for python developer in interview', additional_kwargs={}, response_metadata={})]

In [28]:

ai=llm_with_tools.invoke(message)

In [30]:
ai

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_resume', 'arguments': '{"name": "arav"}'}, '__gemini_function_call_thought_signatures__': {'call_294832': 'EtUECtIEAWkUfRPrNyb7LSdiot4uEGxE/fPDmrZaamvWXwZLei5LjVNxu82KJ0VO64KOpDoeIozYJYKY0uIsWe5XB1SerU9YR9u2r1+yboq5qdSQXXINQoOCZ3ekv3c9gJHuUI7LBfh/iO4dXfKIZTMI2z0pzjzWYcXOlmSKzCV+o8NepAtUaPfH2GOUj2eMtsxk7fYnF4uKSqD0ccw1kTZU9cbTJD4Ar5TIZ2n6Ekti1xQp5O9Th2+9EO9Sj4JnM6d5o6bQI4Y8d+D1j8CHoystFAc8dFq//jAnbI9WIdJPOQJgCurjhwG0DAfeuRYdYo1Rpb8txStkHswr0PnPcny9fy1LGMEC4Mld6fQfZ2FVUZBs+/GtyCvg425oEe2k0MejxTN/3VIkk8q6En0eeCWJo2W3ydDNCVrNTPjxXHkb+RhEs3gar7TijnmmdRTc2rkruZzTPZbaV2pAvtPqJtjCAOt10z1j8ULIAMa/UEGJEg40wgRHZEl1WwrbLOjat7je1twf47ihR3wbbEtwiMuxF/J7XiXZACYVJVPbb9vE9pvVUtDQmC2M3sK1UWp1olZabhg2tzMx4tb4OvqAC0dKoC1ciyPzVfKM5j4LNyHu7S0fTyUsAukCxB3F186/HumKBM0Pkn4qjIrGiahRO6QFDnfW3bfvUycL6CObjs9MAyDTQqPrYwwBtJNR9ZHk2Z9fyZqJ0KataZGqZDUtoFCl5Dlgv9S2zK/bbPfWBVSRJUMFKw99k4z/vUAhx0zZrFm+VtCP5hivg0pONJYFsPGF+B7xHPmQ'}}, response_metadat

In [32]:
ai.tool_calls

[{'name': 'get_resume',
  'args': {'name': 'arav'},
  'id': 'call_294832',
  'type': 'tool_call'}]

In [35]:
tool_name=ai.tool_calls[0]["name"]
tool_ag=ai.tool_calls[0]["args"]["name"]
tool_id=ai.tool_calls[0]["id"]
resume=get_resume.invoke(tool_ag)
print(resume)

Aarav Sharma is a Software Developer based in Nagpur, Maharashtra. 
He can be contacted at aarav.sharma@example.com or 9000000001.
He completed his Bachelor of Technology in Computer Science 
from ABC Institute of Technology in 2024 with 82% marks. 
He has two years of experience working at Tech Solutions Pvt Ltd 
as a Software Developer. His responsibilities include developing 
applications, debugging software, writing database queries, and 
maintaining existing systems. He has technical skills in
 Python, Java, SQL, HTML, CSS, and Git. He developed a Student 
 Management System using Python and MySQL. He is interested in 
 software development and backend programming opportunities.

In [37]:
message.append(ToolMessage(content=resume,tool_call_id=tool_id))
message


[HumanMessage(content='can we select arav for python developer in interview', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_resume', 'arguments': '{"name": "arav"}'}, '__gemini_function_call_thought_signatures__': {'call_294832': 'EtUECtIEAWkUfRPrNyb7LSdiot4uEGxE/fPDmrZaamvWXwZLei5LjVNxu82KJ0VO64KOpDoeIozYJYKY0uIsWe5XB1SerU9YR9u2r1+yboq5qdSQXXINQoOCZ3ekv3c9gJHuUI7LBfh/iO4dXfKIZTMI2z0pzjzWYcXOlmSKzCV+o8NepAtUaPfH2GOUj2eMtsxk7fYnF4uKSqD0ccw1kTZU9cbTJD4Ar5TIZ2n6Ekti1xQp5O9Th2+9EO9Sj4JnM6d5o6bQI4Y8d+D1j8CHoystFAc8dFq//jAnbI9WIdJPOQJgCurjhwG0DAfeuRYdYo1Rpb8txStkHswr0PnPcny9fy1LGMEC4Mld6fQfZ2FVUZBs+/GtyCvg425oEe2k0MejxTN/3VIkk8q6En0eeCWJo2W3ydDNCVrNTPjxXHkb+RhEs3gar7TijnmmdRTc2rkruZzTPZbaV2pAvtPqJtjCAOt10z1j8ULIAMa/UEGJEg40wgRHZEl1WwrbLOjat7je1twf47ihR3wbbEtwiMuxF/J7XiXZACYVJVPbb9vE9pvVUtDQmC2M3sK1UWp1olZabhg2tzMx4tb4OvqAC0dKoC1ciyPzVfKM5j4LNyHu7S0fTyUsAukCxB3F186/HumKBM0Pkn4qjIrGiahRO6QFDnfW3bfvUycL6CObjs9MAyDTQqPrYwwBt

In [38]:
final_result=llm_with_tools.invoke(message)
final_result

AIMessage(content=[{'type': 'text', 'text': 'Yes, **Aarav Sharma** appears to be a strong candidate for selection for a **Python Developer** role. Here is a summary of his qualifications based on his resume:\n\n---\n\n### **Key Qualifications & Strengths**\n\n1. **Relevant Technical Skills:**\n   * **Programming Languages:** **Python**, Java\n   * **Database & Web:** SQL, HTML, CSS\n   * **Version Control:** Git\n\n2. **Practical Experience & Projects:**\n   * **2 years of experience** as a Software Developer at Tech Solutions Pvt Ltd (developing applications, debugging, writing database queries, and system maintenance).\n   * Developed a **Student Management System using Python and MySQL**, demonstrating practical hands-on Python and backend development capability.\n\n3. **Educational Background:**\n   * **B.Tech in Computer Science** from ABC Institute of Technology (2024 graduate with 82% marks).\n\n4. **Career Alignment:**\n   * Primary interests lie in **backend programming** and 

In [39]:
print(final_result.content)

[
    {
        'type': 'text',
        'text': 'Yes, **Aarav Sharma** appears to be a strong candidate for selection for a **Python Developer** 
role. Here is a summary of his qualifications based on his resume:\n\n---\n\n### **Key Qualifications & 
Strengths**\n\n1. **Relevant Technical Skills:**\n   * **Programming Languages:** **Python**, Java\n   * **Database
& Web:** SQL, HTML, CSS\n   * **Version Control:** Git\n\n2. **Practical Experience & Projects:**\n   * **2 years 
of experience** as a Software Developer at Tech Solutions Pvt Ltd (developing applications, debugging, writing 
database queries, and system maintenance).\n   * Developed a **Student Management System using Python and MySQL**, 
demonstrating practical hands-on Python and backend development capability.\n\n3. **Educational Background:**\n   *
**B.Tech in Computer Science** from ABC Institute of Technology (2024 graduate with 82% marks).\n\n4. **Career 
Alignment:**\n   * Primary interests lie in **backend programming** and **software development**.\n\n---\n\n### 
**Recommendation**\nAarav meets the core requirements for a Python Developer position. Evaluating his 
problem-solving skills, code quality, and framework-specific knowledge (e.g., Django, Flask, or FastAPI, depending 
on your stack) during the interview process will help make the final decision.',
        'extras': {
            'signature': 
'EvYJCvMJAWkUfRMBv96AysqWCyIaifbIoqQM1WHGjjHRPEQKP0jSWlkk3pAPvJG6wN4XYOeMHs5VKn9WXyWeu02oew0Hf6iaqrN8fOU4bC9gtqF4Ff
ZXCDvYngF7lLk93VVInQ1WiqFGUJ83vDBZc6M9865AuZvsC5PwXLA/IhcrU29prfT40WY8BtlYHzO6MsBMFKDfD+O1+2T9akZY0m9AWHmeBuVEg9sOD
0/aJKONipv3tARiVTtD6ymdQCaMIRhOuE3WED9qEm/wOyjDVVcuiBGk7RWIKGQn5xZnCDviUU9Y1S/TgHjYIHJQXDIpP7hPKIxYoms17tpizqn0rZt6
xLWy6Mo7Bd/+wuUThz1P4HFIRaXdIKuISQVgoGzm6sDvz4w0VU+T+xU02mMqRwhJ+t5QVsluzMXVbVR2k9dNB/4xsGWkL5ofyM4tSKN4UVSxbw/wV16
qWbXl33DLIDLW//8NehuyPyVuAbexHo7DrxFyUcTT8qvqOHw909c8Z1gwgxXdzE0BEdplWRd/cpC0KdveY1aZ14SqZggbB9jPgof5By4pvTKuOG+bzy
G8cDUdBK7s5l7HCS9rBbruuLyHv+GiUMpAcl8SBybYSlPYbRKqmIpInRfeUumKB9vwT9jiDNp03kk/sVcTBVNmi+o5T2IWTzqqeqdL3wGAHqHySmct0
rQm5Xdsu69+bX4Ix+gsB1Y2YQJ6xyJu5/yx6OgR8T7M9S97PYV8uZ6kqtAU0/mEovQQxop/vt9qw7IQKougBg4oYUgnlVBXfhkndOoQWhwkoX3aeGCc
KYLlA6XAdmhOCI0MYhxsk82jPssaXlhZhsY2/fd8lMUntOSfLwVn1f46lijOxWtZqNHLEd42iKIhHrU/0cUjyEGpJ44f/KEuaB8Gv3hN6iJbYJsNo//
sVi1NAUEVZRZi5F4VRsEV1326Nq/HLED29AGjdTJxFnuueK/A6299zf6eGE9a2ISd7IFdsSMfEaLWDX2dI5KnfA7AzcIywSy1U7EL3HZg8j3DMISegH
m/7P5PA8xIUocoZpiabT+CXdanBGo0KKsf6d25QTwzxw5xEVG7CyZky1lekmzJVsWSJjvC71L4K1b5GTXPVo83no3tCe/oxBVyjemR1GOreJUjbC4TP
SwTWphS5U+u3T0slS+HY3YCBT4mRqna7vDOR2dnPss0d8HJ2W90COpxn/zBFoN+nEeJ7tMjlkm4LLhEHT6XXdbJniqU9A8MeHjU9xW3SE5H5RJ4+an2
swWjeI4JI9VrQTzKC3wgQk4ifHh/0YQFaGe7DONdvcflfR4hlh5JywfdjHrknrsbYyMMBfuG6PBLtv1McpuTF2LGbA4xB2EzuCMh9PzgSm+zYoINQJZ
S/ol1itINWeuYqPCc2bws4H44XWXpg0AiWC+3fYcBcIHisbYZPrtStRtFPPsky9zOnOar6h/WkeLAD1nGJSUz0FfYTQ9Le1KhvEc3T/5gdmyOpmsrmt
3sDNjA3jfLstS+XTAcId/2aD+D0FcTje8adByt6N40VeLjzazYZRrC5vFgeC5dtHEcvmf+NLoXCJ1Nk0XtVyklCPKgYNcrvBWPpcweESIYZRDXIwkof
6a3FwSk0z8Eu8JrbGfFKYi0OlTG7sOvz+jXsrvqGPhSUEj9sHkP4Dm0IVApq4y6Olu7TcKctdPOI0yEbdx1qKG7mg=='
        }
    }
]